# Inspect a Mask Slice

Small standalone helper — **not** part of the main 1→2 pipeline (that's
`xct_membrane_segmentation.ipynb` then `xct_unet_training.ipynb`). Use this
whenever you want to quickly check a single mask file: view it, confirm its
label values are what you expect (e.g. `0/1/2` for a 3-class mask), and
optionally export a viewable image of it.


## How to use this notebook

1. Set `folder` (section 1 below) to wherever your mask `.tif` files live —
   e.g. `full_stack_masks/` (output of `xct_unet_training.ipynb`) or
   `masks_corrected/` (output of `xct_membrane_segmentation.ipynb`).
2. Run section 1 — it picks a random file from that folder, prints its shape,
   dtype, and the first few unique pixel values, and displays it.
3. Re-run section 1 as many times as you like to look at different random
   slices.
4. Optionally run section 2 to save the currently-displayed slice (`img`,
   `files[n]`) as an image — see the two options in that cell for the
   difference between preserving raw label values vs. saving a viewable
   contrast-stretched image.

**Variable to adapt:**

| Variable | What it is |
|---|---|
| `folder` (section 1) | Directory containing the `.tif` mask files to inspect |
| `out_dir` (section 2) | Where to save the exported image, if you use section 2 |


## 1. Load and preview a random mask slice

Picks one `.tif` file at random from `folder` and displays it. The printed
`values` list is the easiest way to sanity-check a mask — for a 3-class
segmentation you should see exactly `[0, 1, 2]`; anything else usually means
you're pointed at a raw image rather than a mask, or the mask has stray
pixel values that need cleaning up.


In [ ]:
from pathlib import Path
import tifffile
import matplotlib.pyplot as plt
import random

# Directory containing the mask .tif files you want to inspect, e.g.
# ".../full_stack_masks/" or ".../masks_corrected/"
folder = Path("/path/to/full_stack")

files = sorted(folder.glob("*.tif"))
assert files, f"No .tif files found in {folder}"

n = random.randint(0, len(files) - 1)  # index of the randomly chosen file
img = tifffile.imread(files[n])
print(f"{files[n].name}  shape={img.shape}  dtype={img.dtype}  "
      f"values={sorted(set(img.ravel().tolist()))[:10]}")

plt.imshow(img, vmin=0, vmax=img.max())
plt.title(files[n].name)
plt.axis("off")
plt.show()


## 2. (Optional) Export the current slice as an image

Saves the slice currently loaded above (`img` / `files[n]`). Two options —
uncomment whichever you need:

- **Option A** writes the raw label values unchanged (e.g. `0/1/2`) — use
  this if the output needs to stay machine-readable (for another script,
  for example). Opening it in a normal image viewer will look almost solid
  black since the values are so close to 0.
- **Option B** saves a contrast-stretched PNG that actually looks like
  something when opened normally — use this for figures, slides, or just
  eyeballing the result. It is **not** safe to reload as label data.


In [ ]:
# Where to save the exported image
out_dir = Path("/home/charlie/usb_drive/data/Masks/segmentation_mask_1/")
out_dir.mkdir(parents=True, exist_ok=True)

# option A -- save the raw pixel data as-is (preserves exact values, e.g. label 0/1/2)
# tifffile.imwrite(out_dir / files[n].name, img)

# option B -- save what you actually see on screen (contrast-stretched PNG for viewing)
plt.imsave(out_dir / f"{files[n].stem}.png", img, cmap="gray")
